# LAMCA-Net Eğitimi: APTOS2019 Diyabetik Retinopati Sınıflandırması
Bu notebook, LAMCA-Net mimarisiyle APTOS2019 veri seti üzerinde eğitim ve değerlendirme sürecini başlatır, tüm metrikleri ve görselleştirmeleri kaydeder.

## 1. Gerekli Kütüphaneleri İçe Aktar
LAMCA-Net, PyTorch ve yardımcı modüller ile çalışır. Ayrıca veri işleme ve görselleştirme için ek kütüphaneler kullanılır.

In [ ]:
# Gerekli kütüphaneleri içe aktar
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from pathlib import Path
from models.lamca_net import LAMCANet
from losses.combined_loss import CombinedLoss
from utils.metrics import compute_metrics, plot_confusion_matrix, plot_metrics
from utils.visualize import TrainingVisualizer
from dataset_loader import APTOS2019DatasetLoader, get_data_loaders

In [ ]:
# GPU kontrolü
if torch.cuda.is_available():
    print(f"CUDA kullanılabilir. GPU adı: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA kullanılabilir değil. Eğitim CPU ile yapılacak.")

## 2. Veri Setini Hazırla
APTOS2019 veri setini yükleyip eğitim, doğrulama ve test olarak ayırıyoruz. Gerekli ön işlemler ve augmentasyonlar uygulanır.

In [38]:
# Veri yolları ve parametreler
batch_size = 16
num_workers = 4

# Orijinal veri loader ile eğitim/val/test loader oluştur
train_loader, val_loader, test_loader, class_weights, (X_train, y_train, X_val, y_val, X_test, y_test) = get_data_loaders(dataset_path="APTOS2019", batch_size=batch_size)

print(f"Eğitim örnekleri: {len(X_train)} | Doğrulama: {len(X_val)} | Test: {len(X_test)}")

total_samples = len(X_train) + len(X_val) + len(X_test)
print(f"Bölünme Oranları -> Eğitim: %{len(X_train)/total_samples*100:.1f} | Doğrulama: %{len(X_val)/total_samples*100:.1f} | Test: %{len(X_test)/total_samples*100:.1f}")

Total training samples: 3662
Class distribution:
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64
Loaded 100/3662 images


KeyboardInterrupt: 

## 3. Modeli Tanımla
LAMCA-Net mimarisi oluşturuluyor ve uygun loss fonksiyonu ile optimizer ayarlanıyor.

In [ ]:
# Model, loss ve optimizer
num_classes = 5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LAMCANet(num_classes=num_classes).to(device)
criterion = CombinedLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
print(model)

## 4. Modeli Eğit
Modeli eğitim verisi üzerinde eğitiyoruz. Her epoch sonunda doğrulama metrikleri ve checkpoint kaydedilir.

In [ ]:
# Eğitim döngüsü
num_epochs = 100  # Daha uzun eğitim için artırıldı
patience = 10  # Erken durdurma için sabır
best_qwk = -np.inf
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': [], 'val_qwk': []}

import os
checkpoint_path = "trained_model/lamca_net_best.pth"
checkpoint_state_path = "trained_model/lamca_net_current_state.pth"
start_epoch = 0

# Checkpoint yükleme (kopma durumunda devam edebilmek için)
if os.path.exists(checkpoint_state_path):
    print("Mevcut checkpoint yükleniyor, eğitime kalındığı yerden devam edilecek...")
    checkpoint = torch.load(checkpoint_state_path)
    model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    start_epoch = checkpoint['epoch'] + 1
    history = checkpoint['history']
    best_qwk = checkpoint['best_qwk']

early_stop_counter = 0

for epoch in range(start_epoch, num_epochs):
    model.train()
    train_loss = 0
    for batch in train_loader:
        images, labels = batch[0].to(device), batch[1].to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels.argmax(1) if labels.ndim > 1 else labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
    train_loss /= len(train_loader.dataset)

    # Validation
    model.eval()
    val_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            images, labels = batch[0].to(device), batch[1].to(device)
            outputs = model(images)
            loss = criterion(outputs, labels.argmax(1) if labels.ndim > 1 else labels)
            val_loss += loss.item() * images.size(0)
            preds = outputs.argmax(1).detach().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy() if labels.ndim == 1 else labels.argmax(1).cpu().numpy())
    val_loss /= len(val_loader.dataset)
    acc, f1, qwk = compute_metrics(all_labels, all_preds)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(acc)
    history['val_f1'].append(f1)
    history['val_qwk'].append(qwk)
    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, Acc={acc:.4f}, F1={f1:.4f}, QWK={qwk:.4f}")
    
    state = {
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'history': history,
        'best_qwk': best_qwk
    }
    torch.save(state, checkpoint_state_path)

    # Checkpoint ve erken durdurma
    if qwk > best_qwk:
        best_qwk = qwk
        torch.save(model.state_dict(), checkpoint_path)
        early_stop_counter = 0
        print(f"--> En iyi model kaydedildi (QWK: {best_qwk:.4f})")
    else:
        early_stop_counter += 1
        if early_stop_counter >= patience:
            print(f"Erken durdurma: {patience} epoch boyunca gelişme olmadı.")
            break

## 5. Eğitim Sonuçlarını Görüntüle
Eğitim ve doğrulama metriklerini, confusion matrix ve başarı grafiklerini çizdiriyoruz. Test seti üzerinde genel başarıyı hesaplıyoruz.

In [ ]:
# Eğitim ve doğrulama metriklerini çizdir
import matplotlib.pyplot as plt
from utils.visualize import TrainingVisualizer
visualizer = TrainingVisualizer(save_dir="trained_model")
visualizer.plot_all_metrics(history)

# En iyi modeli yükle ve test setinde değerlendir
model.load_state_dict(torch.load("trained_model/lamca_net_best.pth"))
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        images, labels = batch[0].to(device), batch[1].to(device)
        outputs = model(images)
        preds = outputs.argmax(1).detach().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy() if labels.ndim == 1 else labels.argmax(1).cpu().numpy())

# Metriklerin Hesaplanması
acc, f1, qwk = compute_metrics(all_labels, all_preds)
print("-----------------------------------------------------")
print(f"Test Sonuçları: Doğruluk (Acc)={acc:.4f}, F1={f1:.4f}, QWK={qwk:.4f}")
print("-----------------------------------------------------")

# Confusion Matrix
class_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(all_labels, all_preds)
visualizer.plot_confusion_matrix(cm, class_names)